In [ ]:
import pandas as pd

pd.read_csv("/home/dylan/projects/ppi/results/toy_all_models/benchmark_manifest.csv")

In [ ]:
pd.read_csv("/home/dylan/projects/ppi/results/toy_all_models/benchmark_summary.csv")

In [ ]:
import pandas as pd

metrics_df = pd.read_csv("results/toy_all_models/metric")
metrics_df

,model_name,features,classifier,k,bm25_k1,bm25_b,max_iter,split_seed,test_size,n_train,...,recall_train_standard_error,f1_train_standard_error,auprc_train_standard_error,auroc_train_standard_error,accuracy_test_standard_error,precision_test_standard_error,recall_test_standard_error,f1_test_standard_error,auprc_test_standard_error,auroc_test_standard_error
0,always_negative,none,always_negative,NaN,NaN,NaN,NaN,0,0.2,800,...,0.0,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000
1,always_positive,none,always_positive,NaN,NaN,NaN,NaN,0,0.2,800,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,binary__linear_svm,binary,linear_svm,3.0,1.5,0.75,1000.0,0,0.2,800,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,binary__logistic,binary,logistic,3.0,1.5,0.75,1000.0,0,0.2,800,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,binary__sgd_logistic,binary,sgd_logistic,3.0,1.5,0.75,1000.0,0,0.2,800,...,0.0,0.000000,0.000000,0.000000,0.002076,0.006919,0.007397,0.005355,0.002303,0.000824
5,bm25__linear_svm,bm25,linear_svm,3.0,1.5,0.75,1000.0,0,0.2,800,...,0.0,0.000000,0.000000,0.000000,0.000232,0.000854,0.000833,0.000586,0.000029,0.000007
6,bm25__logistic,bm25,logistic,3.0,1.5,0.75,1000.0,0,0.2,800,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,bm25__sgd_logistic,bm25,sgd_logistic,3.0,1.5,0.75,1000.0,0,0.2,800,...,0.0,0.001190,0.002241,0.000302,0.002054,0.009136,0.010063,0.007133,0.010254,0.005629
8,count__linear_svm,count,linear_svm,3.0,1.5,0.75,1000.0,0,0.2,800,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,count__logistic,count,logistic,3.0,1.5,0.75,1000.0,0,0.2,800,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [10]:
metrics_df[[col for col in metrics_df.columns if '_mean' in col]]

,accuracy_train_mean,precision_train_mean,recall_train_mean,f1_train_mean,auprc_train_mean,auroc_train_mean,accuracy_test_mean,precision_test_mean,recall_test_mean,f1_test_mean,auprc_test_mean,auroc_test_mean
0,0.797500,NaN,0.000000,0.000000,0.202500,0.500000,0.800000,NaN,0.000000,0.000000,0.200000,0.500000
1,0.202500,0.202500,1.000000,0.336798,0.202500,0.500000,0.200000,0.200000,1.000000,0.333333,0.200000,0.500000
2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.995000,1.000000,0.975000,0.987342,1.000000,1.000000
3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.995000,1.000000,0.975000,0.987342,0.999390,0.999844
4,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.965000,0.928104,0.895833,0.910922,0.975028,0.992286
5,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.985333,0.975235,0.950833,0.962869,0.998807,0.999698
6,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.980000,0.973684,0.925000,0.948718,0.966957,0.987344
7,0.999333,0.996883,1.000000,0.998399,0.997077,0.999608,0.889000,0.805523,0.590833,0.679407,0.695792,0.877240
8,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.995000,1.000000,0.975000,0.987342,1.000000,1.000000
9,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.995000,1.000000,0.975000,0.987342,0.999390,0.999844


In [ ]:
print('hi')

hi


In [ ]:
#!/usr/bin/env python3

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


def read_fasta(path: str) -> dict[str, str]:
    seqs = {}
    current_id = None
    chunks = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if current_id is not None:
                    seqs[current_id] = "".join(chunks)

                current_id = line[1:].split()[0]
                chunks = []
            else:
                chunks.append(line)

    if current_id is not None:
        seqs[current_id] = "".join(chunks)

    return seqs


def check_sequences(df: pd.DataFrame, seqs: dict[str, str]) -> None:
    proteins = set(df["protein_a"]) | set(df["protein_b"])
    missing = sorted(proteins - set(seqs))

    if missing:
        raise ValueError(
            f"{len(missing)} proteins in pairs.csv are missing from FASTA. "
            f"Examples: {missing[:10]}"
        )


def make_pair_features(
    df: pd.DataFrame,
    seqs: dict[str, str],
    vectorizer: TfidfVectorizer,
):
    seq_a = [seqs[p] for p in df["protein_a"]]
    seq_b = [seqs[p] for p in df["protein_b"]]

    xa = vectorizer.transform(seq_a)
    xb = vectorizer.transform(seq_b)

    # Symmetric pair features for undirected PPIs
    x_sum = xa + xb
    x_diff = xa - xb
    x_diff.data = np.abs(x_diff.data)
    x_prod = xa.multiply(xb)

    return hstack([x_sum, x_diff, x_prod]).tocsr()


def safe_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auprc": average_precision_score(y_true, y_prob),
    }

    try:
        metrics["auroc"] = roc_auc_score(y_true, y_prob)
    except ValueError:
        metrics["auroc"] = float("nan")

    return metrics


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--pairs", required=True, help="CSV with protein_a, protein_b, label")
    parser.add_argument("--fasta", required=True, help="FASTA file with protein sequences")
    parser.add_argument("--out", default="predictions.csv")
    parser.add_argument("--metrics-out", default="metrics.csv")
    parser.add_argument("--k", type=int, default=3, help="k-mer size")
    parser.add_argument("--test-size", type=float, default=0.2)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--split-col", default=None, help="Optional column with train/test labels")
    args = parser.parse_args()

    pairs = pd.read_csv(args.pairs)
    seqs = read_fasta(args.fasta)

    required = {"protein_a", "protein_b", "label"}
    missing_cols = required - set(pairs.columns)
    if missing_cols:
        raise ValueError(f"pairs.csv missing columns: {missing_cols}")

    pairs["label"] = pairs["label"].astype(int)
    check_sequences(pairs, seqs)

    if args.split_col and args.split_col in pairs.columns:
        split_values = pairs[args.split_col].astype(str).str.lower()
        train_df = pairs[split_values == "train"].copy()
        test_df = pairs[split_values == "test"].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            raise ValueError("split-col must contain both 'train' and 'test' rows.")
    else:
        train_df, test_df = train_test_split(
            pairs,
            test_size=args.test_size,
            random_state=args.seed,
            stratify=pairs["label"],
        )

    train_proteins = sorted(set(train_df["protein_a"]) | set(train_df["protein_b"]))
    train_sequences = [seqs[p] for p in train_proteins]

    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(args.k, args.k),
        lowercase=False,
    )
    vectorizer.fit(train_sequences)

    x_train = make_pair_features(train_df, seqs, vectorizer)
    x_test = make_pair_features(test_df, seqs, vectorizer)

    y_train = train_df["label"].values
    y_test = test_df["label"].values

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear",
        random_state=args.seed,
    )
    model.fit(x_train, y_train)

    y_prob = model.predict_proba(x_test)[:, 1]
    metrics = safe_metrics(y_test, y_prob)

    pred_df = test_df[["protein_a", "protein_b", "label"]].copy()
    pred_df["pred_proba"] = y_prob
    pred_df["pred_label"] = (y_prob >= 0.5).astype(int)
    pred_df.to_csv(args.out, index=False)

    metrics_df = pd.DataFrame(
        [
            {
                "n_train": len(train_df),
                "n_test": len(test_df),
                "k": args.k,
                **metrics,
            }
        ]
    )
    metrics_df.to_csv(args.metrics_out, index=False)

    print(metrics_df.to_string(index=False))
    print(f"\nSaved predictions to: {args.out}")
    print(f"Saved metrics to: {args.metrics_out}")


if __name__ == "__main__":
    main()